# 03 — Nettoyage multilingue et découpage

Normalisation prudente : on réduit le bruit sans modifier la portée juridique du texte. Le texte original reste conservé à côté du texte normalisé.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
INDEX = DATA / "index"
for directory in (RAW, PROCESSED, INDEX):
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

In [ ]:
import json
import re
import unicodedata

ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652]")

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = ARABIC_DIACRITICS.sub("", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def chunk_words(text: str, size: int = 260, overlap: int = 50) -> list[str]:
    words = text.split()
    if not words:
        return []
    step = max(1, size - overlap)
    return [" ".join(words[start:start + size]) for start in range(0, len(words), step)]

pages_path = PROCESSED / "pages.jsonl"
pages = []
if pages_path.exists():
    pages = [json.loads(line) for line in pages_path.read_text(encoding="utf-8").splitlines()]

chunks = []
for page in pages:
    normalized = normalize_text(page["text"])
    for chunk_id, chunk in enumerate(chunk_words(normalized)):
        chunks.append({**page, "chunk_id": chunk_id, "text_normalized": chunk})

chunks_path = PROCESSED / "chunks.jsonl"
with chunks_path.open("w", encoding="utf-8") as stream:
    for row in chunks:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")
{"pages": len(pages), "chunks": len(chunks), "output": str(chunks_path)}